In [ ]:
import os
from dotenv import load_dotenv
import torch
import json
import subprocess
import sys
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
from difflib import SequenceMatcher
import re

env = "GPU"
if env == "GPU":
    os.environ["HF_HOME"] = rf"/cs/student/project_msc/2025/dsml/navnmann/huggingface"
    load_dotenv(rf"/cs/student/project_msc/2025/dsml/navnmann/MSc_code/.env")

# Theme Knowledge Graph Extraction

Distil the DeepSeek gold labels into Qwen2.5-7B-Instruct

The student is trained on the **same system prompt** the teacher used (minus its
predefined themes step which is out of scope here) so the target is exactly the teacher's output format:

```json
{"knowledge_graph": [
  {"triplet": ["head", "HEAD_TYPE", "RELATION", "tail", "TAIL_TYPE"],
   "theme": "...", "dimension": "..."}
]}
```

Each eval/train stage loads the 4 bit base model and frees it before the next, so you
never hold two models in VRAM at once. Every metric caches to JSON, so an interrupted
run never repeats finished work.

**Sized for a 4090 (24GB)**: MAX_LENGTH=6144, 12k char articles

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # strong open 7B for structured extraction

if env == "GPU":
    _ROOT = "/cs/student/project_msc/2025/dsml/navnmann/MSc_code"
else:
    _ROOT = os.path.abspath("..")

RUN_DIR = f"{_ROOT}/llm_finetune_output"
SOURCE_CSV = f"{_ROOT}/data/combined_2017_2023_themes_with_KG.csv"

OUT_DIR = f"{RUN_DIR}/kg-lora"  # adapter + training checkpoints
TRAIN_PATH = f"{RUN_DIR}/train.jsonl"
VAL_PATH = f"{RUN_DIR}/val.jsonl"
TEST_PATH = f"{RUN_DIR}/test.jsonl"
SPLIT_PATH = f"{RUN_DIR}/split.json"

# training
EPOCHS = 3.0
LR = 2e-4
MAX_LENGTH = 6144
BATCH_SIZE = 1
GRAD_ACCUM = 20
LORA_R = 16

# eval/data
MAX_NEW_TOKENS = 4096
MAX_ARTICLE_CHARS = 12000
THEME_FUZZY_THRESHOLD = 0.85  # 1.0 = exact theme match; lower = more lenient

TEST_SIZE = 200
VAL_SIZE = 200
TOPIC_TEST_QUOTA = 8
CURVE_EVAL_N = 50

SEED = 72
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = None

# gold ontology copied from notebooks/extract_themes_KG_LLM.ipynb
ENTITY_TYPES = {
    "ORG",
    "ORG/GOV",
    "ORG/REG",
    "PERSON",
    "GPE",
    "COMP",
    "PRODUCT",
    "EVENT",
    "SECTOR",
    "ECON_INDICATOR",
    "FIN_INSTRUMENT",
    "CONCEPT",
}
THEME_RELATIONS = {"HAS_ACTOR", "HAS_TARGET", "HAS_CONTEXT", "AFFECTS"}
ENTITY_RELATIONS = {"ACTS_ON", "BELONGS_TO"}
ALL_RELATIONS = THEME_RELATIONS | ENTITY_RELATIONS
DIMENSIONS = {
    "economic_monetary_event",
    "geopolitical_factor",
    "sector_or_industry",
    "policy_or_regulation",
    "technology_concept",
    "macro_trend",
}
PREDEFINED_THEMES = [
    "AI",
    "Covid19 pandemic",
    "USA Tariff",
    "Russia - Ukraine war",
    "Oil Crisis",
    "Semiconductor Chips",
    "Fed interest rates",
]

os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"RUN_DIR {RUN_DIR}")
print(f"SOURCE_CSV {SOURCE_CSV} (exists: {os.path.exists(SOURCE_CSV)})")

In [ ]:
def save_json(obj, path):

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

    return path


_cfg = dict(
    model_name=MODEL_NAME,
    seed=SEED,
    epochs=EPOCHS,
    lr=LR,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    lora_r=LORA_R,
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    max_new_tokens=MAX_NEW_TOKENS,
    max_article_chars=MAX_ARTICLE_CHARS,
    theme_fuzzy_threshold=THEME_FUZZY_THRESHOLD,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    curve_eval_n=CURVE_EVAL_N,
    source_csv=SOURCE_CSV,
    train_path=TRAIN_PATH,
    val_path=VAL_PATH,
    test_path=TEST_PATH,
    out_dir=OUT_DIR,
)
print("saved", save_json(_cfg, f"{RUN_DIR}/config.json"))

# it is good to lock in current env for finetuning
lock = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True
).stdout

with open(f"{RUN_DIR}/requirements_lock.txt", "w") as f:
    f.write(lock)

print("saved", f"{RUN_DIR}/requirements_lock.txt")

### Prompt & target construction

The same prompt is used for training **and** both evals, so the before/after comparison is fair.

In [ ]:
# Copied from notebooks/extract_themes_KG_LLM.ipynb —-> the prompt that produced the gold labels
SYSTEM_PROMPT = """You are an expert financial news analyst and knowledge graph extractor.
 
Given the news article below, produce a single JSON object with exactly 
one key: "knowledge_graph".
 
Process in this strict order:
  Step 1 → Identify themes          (working step — not output)
  Step 2 → Build knowledge_graph anchored to those themes
 
═══════════════════════════════════════════════════
STEP 1 — IDENTIFY THEMES
═══════════════════════════════════════════════════
A theme is a TOPIC TAG — the kind of label a financial analyst would file 
the article under. It is not a fact, and it is not a name.
 
Two tests. A phrase must pass both:
  TEST 1 — Could this same phrase tag a different article, six months from 
           now, about different companies? If no, it is a fact. Reject it.
  TEST 2 — Does it contain a company, person, place, product, or number? 
           If yes, it is an entity. Reject it.
 
Shape: 2–5 words. A noun phrase. Understandable on its own.
 
Go through the 6 dimensions below. For each one, write ONE theme phrase if 
that dimension is present in the article — otherwise skip it. Do not force 
a phrase. Most articles will yield 2–3 themes, not 6.
 
  DIMENSION                  GOOD                            BAD
  ─────────────────────────  ──────────────────────────────  ─────────────────────────
  economic_monetary_event    "central bank rate tightening"  "Fed hiked 50bps"
  geopolitical_factor        "cross border trade tension"    "Russia invaded Ukraine"
  sector_or_industry         "semiconductor supply shortage" "TSMC fab delay"
  policy_or_regulation       "import tariff escalation"      "Section 301 tariffs"
  technology_concept         "large language model adoption" "GPT-4 launch"
  macro_trend                "persistent inflation pressure" "CPI rose 3.2%"
 
The BAD column fails because each one names a specific company, person, 
number, or one-off occurrence. The GOOD column would still make sense as a 
tag on an article you have never seen.
 
Write each theme phrase down once, exactly as you will use it. You will 
repeat it verbatim in every triplet in Step 2 — same words, same order, 
same spelling, every time. Do not shorten or reword it later.
 
═══════════════════════════════════════════════════
STEP 2 — KNOWLEDGE GRAPH
═══════════════════════════════════════════════════
The knowledge graph exists to describe the themes from Step 1 — nothing else.
Every triplet is built AROUND one of those themes and must trace back to it,
directly or in one hop. If a fact in the article does not connect to any
discovered theme, leave it out. No free-floating facts; no theme without at
least one triplet.
 
Take the themes from Step 1 one at a time. For each theme:
 
  a) The theme phrase is a node of type CONCEPT
  b) Link the theme to real-world entities from the article using the 
     theme-to-entity relationships below
  c) Expand outward: add entity-to-entity triplets that explain why that 
     theme matters in the article
  d) Drop any triplet that cannot be connected — directly or in one hop — 
     back to a theme
  e) Tag every triplet with its owning theme phrase and dimension
 
Finish one theme completely before starting the next.
 
RELATIONSHIPS — use only these:
 
  Theme-to-entity (the theme is always the HEAD):
    HAS_ACTOR     → who initiates or drives the theme
    HAS_TARGET    → what the theme is directed at
    HAS_CONTEXT   → the sector, region, indicator, instrument, or product 
                    the theme operates in
    AFFECTS       → who or what the theme impacts
 
  Entity-to-entity (support, one hop from the theme):
    ACTS_ON       → one entity does something to another (announces, 
                    introduces, controls, invests in, impacts)
    BELONGS_TO    → membership, ownership, sector placement, or location
 
ENTITY TYPES — use exactly one per entity:
  ORG           → Organizations (non-government, non-regulatory)
  ORG/GOV       → Government bodies
  ORG/REG       → Regulatory bodies
  PERSON        → Individuals
  GPE           → Countries, cities, geopolitical entities
  COMP          → Companies
  PRODUCT       → Products or services
  EVENT         → Specific material events
  SECTOR        → Industries or company sectors
  ECON_INDICATOR → Economic indicators (not raw numbers/percentages)
  FIN_INSTRUMENT → Financial instruments, markets
  CONCEPT       → Theme anchor nodes ONLY — never create a CONCEPT node 
                  that is not a theme phrase from Step 1
 
CONSTRAINTS:
  • No generic, numerical, or temporal entities
  • No redundant triplets
  • No strictly past-tense events
  • Disambiguate entities (e.g. "BOE" → "Bank of England")
  • Max 4 words per entity label
  • Every triplet must belong to exactly one discovered theme — either a
    theme-to-entity edge, or an entity-to-entity edge one hop from that theme
  • The ONLY CONCEPT nodes allowed are the Step 1 theme phrases; never invent
    an extra CONCEPT node (e.g. do not link one theme to another new concept)
  • Give every discovered theme at least one triplet
 
═══════════════════════════════════════════════════
OUTPUT FORMAT — return ONLY valid JSON, no extra text
═══════════════════════════════════════════════════
Do not output the Step 1 theme list as its own key. Themes appear only in 
the "theme" field of each triplet.
 
{
  "knowledge_graph": [
    {
      "triplet": ["head_entity", "head_type", "relationship", "tail_entity", "tail_type"],
      "theme": "the Step 1 theme phrase — identical in every triplet that belongs to it",
      "dimension": "economic_monetary_event | geopolitical_factor | sector_or_industry | policy_or_regulation | technology_concept | macro_trend"
    }
  ]
}

═══════════════════════════════════════════════════
INPUT_TEXT:
"""


def build_prompt_messages(article):
    # The teacher passed the bare article as the user message
    article = article[:MAX_ARTICLE_CHARS]

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": article},
    ]


def serialize_target(knowledge_graph):
    return json.dumps({"knowledge_graph": knowledge_graph}, ensure_ascii=False)


def to_chat_example(row):

    return {
        "prompt": build_prompt_messages(row["article"]),
        "completion": [
            {
                "role": "assistant",
                "content": serialize_target(row["knowledge_graph"]),
            }
        ],
    }

## Data loading

JSONL, one object per line, written by the prep cell below:

```json
{"article_id": "f35f8e6d0250",
 "article": "Apple said it is rethinking the Mac Pro...",
 "knowledge_graph": [
   {"triplet": ["product update transparency", "CONCEPT", "HAS_ACTOR", "Apple", "COMP"],
    "theme": "product update transparency",
    "dimension": "technology_concept"}
 ],
 "themes": [{"theme": "product update transparency", "dimension": "technology_concept"}]}
```

In [ ]:
def extract_themes(kg):

    themes, seen = [], set()

    for t in kg:
        pair = (t["theme"], t["dimension"])

        if pair not in seen:
            seen.add(pair)
            themes.append({"theme": t["theme"], "dimension": t["dimension"]})

    return themes


def load_pairs(path, ids=None):

    rows = []
    with open(path, encoding="utf-8") as f:

        for i, line in enumerate(f):

            line = line.strip()
            if not line:
                continue

            obj = json.loads(line)
            article = obj["article"]
            kg = obj.get("knowledge_graph", [])

            if not isinstance(kg, list):
                raise ValueError(f"line {i}: knowledge_graph must be a list")

            article_id = obj.get("article_id")
            if ids is not None and article_id not in ids:
                continue

            rows.append(
                {
                    "article_id": article_id,
                    "article": article,
                    "knowledge_graph": kg,
                    "themes": obj.get("themes") or extract_themes(kg),
                }
            )

    return rows

### Build the JSONL splits from the gold CSV

In [ ]:
_ASSISTANT_OVERHEAD = 8


def _prompt_token_len(tok, article):
    text = tok.apply_chat_template(
        build_prompt_messages(article), tokenize=False, add_generation_prompt=True
    )

    return len(tok(text, add_special_tokens=False)["input_ids"])


def _target_token_len(tok, kg):
    return len(tok(serialize_target(kg), add_special_tokens=False)["input_ids"])


def fit_target(tok, article, kg, budget=None):

    budget = budget or MAX_LENGTH
    room = budget - _prompt_token_len(tok, article) - _ASSISTANT_OVERHEAD

    if room <= 0:
        return [], len(kg)

    if _target_token_len(tok, kg) <= room:
        return kg, 0
    lo, hi = 0, len(kg)  # invariant: kg[:lo] fits

    while lo < hi:
        mid = (lo + hi + 1) // 2
        if _target_token_len(tok, kg[:mid]) <= room:
            lo = mid
        else:
            hi = mid - 1

    return kg[:lo], len(kg) - lo


def _year_stratified(df, n, year_col="year"):

    if n <= 0 or df.empty:
        return df.iloc[0:0]

    n = min(n, len(df))
    frac = df[year_col].value_counts(normalize=True)
    picks = []

    for year, group in df.groupby(year_col):
        k = max(1, round(frac[year] * n))
        picks.append(group.sample(n=min(k, len(group)), random_state=SEED))

    return pd.concat(picks).sample(frac=1, random_state=SEED).head(n)


def _topic_quota(df, quota, exclude_ids):

    taken = set(exclude_ids)
    picks = []
    for theme in PREDEFINED_THEMES:

        mask = df["_pos_themes"].apply(lambda s: theme in s)
        cand = df[mask & ~df["article_id"].isin(taken)]

        if cand.empty:
            print(f"{theme}: no candidates")
            continue

        chosen = cand.sample(n=min(quota, len(cand)), random_state=SEED)
        picks.append(chosen)
        taken.update(chosen["article_id"])
        print(f"{theme}: {len(chosen)}")

    return pd.concat(picks) if picks else df.iloc[0:0]


def _write_jsonl(path, frame):

    with open(path, "w", encoding="utf-8") as f:

        for _, r in frame.iterrows():
            f.write(
                json.dumps(
                    {
                        "article_id": r["article_id"],
                        "article": r["news_text"],
                        "knowledge_graph": r["kg"],
                        "themes": extract_themes(r["kg"]),
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

    return path


def build_splits():

    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)

    df = pd.read_csv(
        SOURCE_CSV,
        usecols=[
            "article_id",
            "year",
            "description",
            "maintext",
            "knowledge_graph",
            "predefined_themes",  # stratification key only
        ],
    )
    print(f"CSV rows: {len(df):,}")

    df = df[df["knowledge_graph"].notna()].copy()
    print(f"with gold KG: {len(df):,}")

    # identical construction to the teacher run, so the student sees the same text
    df["news_text"] = (
        df["description"].fillna("").str.strip()
        + "\n\n"
        + df["maintext"].fillna("").str.strip()
    ).str.strip()

    df["kg"] = df["knowledge_graph"].apply(json.loads)
    df = df[df["kg"].apply(len) > 0].copy()
    print(f"with non-empty KG: {len(df):,}")

    # drop examples that don't fit the budget WHOLE
    seq_lens = [
        _prompt_token_len(tok, article)
        + _target_token_len(tok, kg)
        + _ASSISTANT_OVERHEAD
        for article, kg in zip(df["news_text"], df["kg"])
    ]
    df["_seq_len"] = seq_lens
    n_before, trip_before = len(df), int(df["kg"].apply(len).sum())
    df = df[df["_seq_len"] <= MAX_LENGTH].copy()
    trip_after = int(df["kg"].apply(len).sum())
    print(
        f"Dropped examples over MAX_LENGTH={MAX_LENGTH}: "
        f"{n_before - len(df):,}/{n_before:,} examples "
        f"({(n_before - len(df)) / n_before:.1%}), carrying "
        f"{trip_before - trip_after:,}/{trip_before:,} triplets "
        f"({(trip_before - trip_after) / max(trip_before, 1):.1%})"
    )
    print(f"kept: {len(df):,}   max seq {int(df['_seq_len'].max()):,} tok")

    for article, kg in zip(df["news_text"].head(100), df["kg"].head(100)):
        _, n_drop = fit_target(tok, article, kg)
        assert n_drop == 0, "kept example still over budget - budget math drifted"

    tgt_max = int(max(_target_token_len(tok, kg) for kg in df["kg"]))
    print(
        f"longest kept gold target: {tgt_max:,} tok  (MAX_NEW_TOKENS={MAX_NEW_TOKENS})"
    )
    assert (
        tgt_max <= MAX_NEW_TOKENS
    ), "raise MAX_NEW_TOKENS above the longest gold target"

    df["_pos_themes"] = df["predefined_themes"].apply(
        lambda s: (
            {k for k, v in json.loads(s).items() if v == "Yes"}
            if isinstance(s, str)
            else set()
        )
    )

    print("Test set topic quota")
    topic = _topic_quota(df, TOPIC_TEST_QUOTA, exclude_ids=set())
    rest = df[~df["article_id"].isin(topic["article_id"])]
    test = pd.concat([topic, _year_stratified(rest, TEST_SIZE - len(topic))])

    rest = df[~df["article_id"].isin(test["article_id"])]
    val = _year_stratified(rest, VAL_SIZE)
    train = rest[~rest["article_id"].isin(val["article_id"])]

    assert not (set(train["article_id"]) & set(test["article_id"]))
    assert not (set(train["article_id"]) & set(val["article_id"]))
    assert not (set(val["article_id"]) & set(test["article_id"]))

    _write_jsonl(TRAIN_PATH, train)
    _write_jsonl(VAL_PATH, val)
    _write_jsonl(TEST_PATH, test)

    curve_ids = test["article_id"].tolist()[:CURVE_EVAL_N]
    save_json(
        {
            "train": train["article_id"].tolist(),
            "val": val["article_id"].tolist(),
            "test": test["article_id"].tolist(),
            "curve_eval": curve_ids,
        },
        SPLIT_PATH,
    )

    print(f"Train {len(train):,}; val {len(val):,}; test {len(test):,}")
    print(f"curve eval subset: {len(curve_ids)} articles (frozen in {SPLIT_PATH})")
    dims = pd.Series(
        [t["dimension"] for kg in test["kg"] for t in extract_themes(kg)]
    ).value_counts()
    print(f"Test dimensions covered: {len(dims)}/6 -> {dims.to_dict()}")


if all(os.path.exists(p) for p in (TRAIN_PATH, VAL_PATH, TEST_PATH, SPLIT_PATH)):
    print("Splits already exist; skipping ... Delete them in RUN_DIR to rebuild.")
else:
    build_splits()

CURVE_EVAL_IDS = json.load(open(SPLIT_PATH))["curve_eval"]

### Output parsing

In [ ]:
def _first_json_object(text):
    start = text.find("{")

    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):

        if text[i] == "{":
            depth += 1

        elif text[i] == "}":
            depth -= 1

            if depth == 0:
                return text[start : i + 1]

    return None


def parse_model_output(text):

    if text is None:
        return None, False

    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    candidate = _first_json_object(text)

    if candidate is None:
        return None, False

    try:
        obj = json.loads(candidate)

    except json.JSONDecodeError:
        return None, False

    if not isinstance(obj, dict):
        return None, False

    clean = []
    kg = obj.get("knowledge_graph", [])
    if isinstance(kg, list):

        for entry in kg:

            if not isinstance(entry, dict):
                continue
            triplet = entry.get("triplet")

            if not isinstance(triplet, (list, tuple)) or len(triplet) != 5:
                continue

            if not all(isinstance(x, str) and x.strip() for x in triplet):
                continue
            theme = entry.get("theme")

            if not (isinstance(theme, str) and theme.strip()):
                continue

            clean.append(
                {
                    "triplet": [x.strip() for x in triplet],
                    "theme": theme.strip(),
                    "dimension": str(entry.get("dimension", "")).strip().lower(),
                }
            )

    return {"knowledge_graph": clean, "themes": extract_themes(clean)}, True

### Metrics

In [ ]:
SCORED_FIELDS = ("theme", "theme_dim", "triplet", "triplet_untyped", "entity")


def _norm(s):
    return re.sub(r"\s+", " ", str(s).strip().lower())


def _fuzzy_tp(pred, gold, threshold):

    gold_left = list(gold)
    tp = 0

    for p in pred:
        best_i, best_r = -1, 0.0

        for i, g in enumerate(gold_left):
            r = 1.0 if p == g else SequenceMatcher(None, p, g).ratio()
            if r > best_r:
                best_r, best_i = r, i

        if best_i >= 0 and best_r >= threshold:
            tp += 1
            gold_left.pop(best_i)

    return tp


def _fuzzy_tp_by_dim(pred_pairs, gold_pairs, threshold):

    tp = 0
    for d in {d for _, d in pred_pairs} | {d for _, d in gold_pairs}:
        tp += _fuzzy_tp(
            [t for t, dd in pred_pairs if dd == d],
            [t for t, dd in gold_pairs if dd == d],
            threshold,
        )

    return tp


def _triplet_key(t):
    return " ||| ".join(_norm(x) for x in t)


def _untyped_key(t):
    return " ||| ".join(_norm(x) for x in (t[0], t[2], t[3]))


def _entities(kg):

    out = set()
    for e in kg:
        head, htype, _, tail, ttype = e["triplet"]

        if htype.upper() != "CONCEPT":
            out.add(_norm(head))

        if ttype.upper() != "CONCEPT":
            out.add(_norm(tail))

    return out


def is_valid_triplet(entry, article_themes):

    head, htype, rel, tail, ttype = entry["triplet"]

    htype, ttype, rel = htype.upper(), ttype.upper(), rel.upper()
    if htype not in ENTITY_TYPES or ttype not in ENTITY_TYPES:
        return False

    if rel not in ALL_RELATIONS:
        return False

    if entry["dimension"] not in DIMENSIONS:
        return False

    # theme-to-entity edges must touch the theme's CONCEPT node
    if rel in THEME_RELATIONS and "CONCEPT" not in (htype, ttype):
        return False

    # the only CONCEPT nodes allowed are this article's own theme phrases
    if htype == "CONCEPT" and _norm(head) not in article_themes:
        return False

    if ttype == "CONCEPT" and _norm(tail) not in article_themes:
        return False

    if _norm(head) == _norm(tail):
        return False

    return True


def score_example(pred, gold, theme_threshold=None):

    if theme_threshold is None:
        theme_threshold = THEME_FUZZY_THRESHOLD

    if pred is None:
        pred = {"knowledge_graph": [], "themes": []}

    p_kg, g_kg = pred["knowledge_graph"], gold["knowledge_graph"]

    p_pairs = [(_norm(t["theme"]), _norm(t["dimension"])) for t in pred["themes"]]
    g_pairs = [(_norm(t["theme"]), _norm(t["dimension"])) for t in gold["themes"]]
    pt = [t for t, _ in p_pairs]
    gt = [t for t, _ in g_pairs]

    p_tr = {_triplet_key(e["triplet"]) for e in p_kg}
    g_tr = {_triplet_key(e["triplet"]) for e in g_kg}
    p_un = {_untyped_key(e["triplet"]) for e in p_kg}
    g_un = {_untyped_key(e["triplet"]) for e in g_kg}
    p_ent, g_ent = _entities(p_kg), _entities(g_kg)

    p_themes = {t for t, _ in p_pairs}
    n_ok = sum(is_valid_triplet(e, p_themes) for e in p_kg)

    return {
        "theme": (_fuzzy_tp(pt, gt, theme_threshold), len(pt), len(gt)),
        "theme_dim": (
            _fuzzy_tp_by_dim(p_pairs, g_pairs, theme_threshold),
            len(p_pairs),
            len(g_pairs),
        ),
        "triplet": (len(p_tr & g_tr), len(p_tr), len(g_tr)),
        "triplet_untyped": (len(p_un & g_un), len(p_un), len(g_un)),
        "entity": (len(p_ent & g_ent), len(p_ent), len(g_ent)),
        "ontology": (n_ok, len(p_kg)),
        "exact_match": (set(pt) == set(gt)) and (p_tr == g_tr),
    }


def _prf(tp, pn, gn):

    p = tp / pn if pn else 0.0
    r = tp / gn if gn else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0

    return p, r, f


def aggregate(per_example, n_valid_json):

    out = {}
    n = len(per_example)
    for field in SCORED_FIELDS:

        tp = sum(e[field][0] for e in per_example)
        pn = sum(e[field][1] for e in per_example)
        gn = sum(e[field][2] for e in per_example)
        p, r, f = _prf(tp, pn, gn)
        out[field] = {"precision": p, "recall": r, "f1": f}

    n_ok = sum(e["ontology"][0] for e in per_example)
    n_tr = sum(e["ontology"][1] for e in per_example)
    out["ontology_valid_rate"] = n_ok / n_tr if n_tr else 0.0
    out["n_pred_triplets"] = n_tr
    out["json_valid_rate"] = n_valid_json / n if n else 0.0
    out["exact_match_rate"] = (
        sum(e["exact_match"] for e in per_example) / n if n else 0.0
    )
    out["n_examples"] = n

    return out


def print_report(metrics, title="Results"):

    print(f"=== {title} ===")
    print(f"Examples: {metrics['n_examples']}")
    print(f"Valid JSON rate: {metrics['json_valid_rate']:.3f}")
    print(
        f"Ontology valid rate: {metrics['ontology_valid_rate']:.3f}"
        f"({metrics['n_pred_triplets']:,} predicted triplets)"
    )
    print(f"Exact match rate: {metrics['exact_match_rate']:.3f}")

    for field in SCORED_FIELDS:

        m = metrics[field]
        print(
            f"{field:<15} P / R / F1:"
            f"{m['precision']:.3f} / {m['recall']:.3f} / {m['f1']:.3f}"
        )

In [ ]:
_gold_kg = [
    {
        "triplet": [
            "import tariff escalation",
            "CONCEPT",
            "HAS_ACTOR",
            "EU",
            "ORG/GOV",
        ],
        "theme": "import tariff escalation",
        "dimension": "policy_or_regulation",
    },
    {
        "triplet": ["EU", "ORG/GOV", "ACTS_ON", "steel exporters", "SECTOR"],
        "theme": "import tariff escalation",
        "dimension": "policy_or_regulation",
    },
]

_raw = (
    "```json"
    + json.dumps(
        {
            "knowledge_graph": [
                {
                    "triplet": [
                        "Import Tariff Escalation",
                        "CONCEPT",
                        "HAS_ACTOR",
                        " EU ",
                        "ORG/GOV",
                    ],
                    "theme": "Import Tariff Escalation",
                    "dimension": "Policy_Or_Regulation",
                },
                {
                    "triplet": [
                        "EU",
                        "ORG/GOV",
                        "ACTS_ON",
                        "steel exporters",
                        "SECTOR",
                    ],
                    "theme": "Import Tariff Escalation",
                    "dimension": "policy_or_regulation",
                },
            ]
        }
    )
    + "\n```"
)

_pred, _valid = parse_model_output(_raw)
_gold = {"knowledge_graph": _gold_kg, "themes": extract_themes(_gold_kg)}
_m = aggregate([score_example(_pred, _gold)], int(_valid))
print_report(_m, "self test should be all 1.000")
assert _valid and _m["exact_match_rate"] == 1.0, "parser/metric mismatch"
assert all(
    _m[f]["f1"] == 1.0 for f in SCORED_FIELDS
), "F1 not perfect on a perfect pred"
assert _m["ontology_valid_rate"] == 1.0

# and the ontology check must actually reject things
_bad = [
    {  # NOT_A_REL is not in ALL_RELATIONS
        "triplet": ["EU", "ORG/GOV", "NOT_A_REL", "steel exporters", "SECTOR"],
        "theme": "import tariff escalation",
        "dimension": "policy_or_regulation",
    },
    {  # a CONCEPT node that is not one of this article's themes
        "triplet": ["global inflation surge", "CONCEPT", "HAS_ACTOR", "EU", "ORG/GOV"],
        "theme": "import tariff escalation",
        "dimension": "policy_or_regulation",
    },
    {  # HAS_ACTOR is a theme relation but neither end is a CONCEPT
        "triplet": ["EU", "ORG/GOV", "HAS_ACTOR", "steel exporters", "SECTOR"],
        "theme": "import tariff escalation",
        "dimension": "policy_or_regulation",
    },
]

_themes = {"import tariff escalation"}
assert not any(is_valid_triplet(e, _themes) for e in _bad), "validator too lenient"
print("Ontology validator rejected all 3 bad triplets")

### Staged scoring; theme extraction and KG construction

In [ ]:
STAGED_PRF_FIELDS = (
    "theme_exact",
    "theme_structural",
    "triplet_aligned",
    "triplet_untyped_aligned",
    "entity_norm",
    "aligned_subgraph",
)
CONDITIONAL_FIELDS = (
    "relation_given_pair",
    "types_given_pair",
    "relation_types_given_pair",
)
_CORP_SUFFIX = re.compile(
    r"\b(inc|corp|corporation|ltd|limited|plc|co|holdings?|group|llc|lp|sa|ag|nv|se)\b\.?"
)


def _norm_entity(s):

    base = _norm(s)
    out = re.sub(r"^the\s+", "", base)
    out = _CORP_SUFFIX.sub(" ", out)
    out = re.sub(r"[^a-z0-9 ]", " ", out)
    out = re.sub(r"\s+", " ", out).strip()

    return out or base


def _entity_triplet_key(t):
    return " ||| ".join(
        [_norm_entity(t[0]), _norm(t[1]), _norm(t[2]), _norm_entity(t[3]), _norm(t[4])]
    )


def _entity_untyped_key(t):
    return " ||| ".join([_norm_entity(t[0]), _norm(t[2]), _norm_entity(t[3])])


def _nodes(triplets):

    out = set()
    for t in triplets:

        if t[1].upper() != "CONCEPT":
            out.add(_norm_entity(t[0]))

        if t[4].upper() != "CONCEPT":
            out.add(_norm_entity(t[3]))

    return out


def _multiset(keys):

    c = {}
    for k in keys:
        c[k] = c.get(k, 0) + 1

    return c


def _consume(pool, key):

    if pool.get(key, 0) > 0:
        pool[key] -= 1
        return True

    return False


def _relabel(triplet, old_theme, new_theme):

    t = list(triplet)

    if t[1].upper() == "CONCEPT" and _norm(t[0]) == _norm(old_theme):
        t[0] = new_theme

    if t[4].upper() == "CONCEPT" and _norm(t[3]) == _norm(old_theme):
        t[3] = new_theme

    return t


def align_themes(p_kg, g_kg):

    p_themes = sorted({e["theme"] for e in p_kg})
    g_themes = sorted({e["theme"] for e in g_kg})

    if not p_themes or not g_themes:
        return {}

    gold_pool = _multiset(_triplet_key(e["triplet"]) for e in g_kg)
    overlap = np.zeros((len(p_themes), len(g_themes)), dtype=int)

    for i, p in enumerate(p_themes):
        p_tr = [e["triplet"] for e in p_kg if e["theme"] == p]

        for j, g in enumerate(g_themes):
            pool = dict(gold_pool)
            overlap[i, j] = sum(
                _consume(pool, _triplet_key(_relabel(t, p, g))) for t in p_tr
            )

    rows, cols = linear_sum_assignment(-overlap)

    return {p_themes[i]: g_themes[j] for i, j in zip(rows, cols) if overlap[i, j] > 0}


def score_staged(pred, gold):

    p_kg = pred["knowledge_graph"] if pred else []
    g_kg = gold["knowledge_graph"]
    g_tr = [e["triplet"] for e in g_kg]

    mapping = align_themes(p_kg, g_kg)
    p_tr = [
        _relabel(e["triplet"], e["theme"], mapping.get(e["theme"], e["theme"]))
        for e in p_kg
    ]

    p_themes = [_norm(t["theme"]) for t in (pred["themes"] if pred else [])]
    g_themes = [_norm(t["theme"]) for t in gold["themes"]]
    n_p, n_g = len(p_themes), len(g_themes)
    tp_exact = len(set(p_themes) & set(g_themes))

    pool = _multiset(_entity_triplet_key(t) for t in g_tr)
    tp_tr = sum(_consume(pool, _entity_triplet_key(t)) for t in p_tr)

    pool = _multiset(_entity_untyped_key(t) for t in g_tr)
    tp_un = sum(_consume(pool, _entity_untyped_key(t)) for t in p_tr)

    p_nodes, g_nodes = _nodes(p_tr), _nodes(g_tr)

    a_p = {p for p in mapping}
    a_g = set(mapping.values())
    p_sub = [
        _relabel(e["triplet"], e["theme"], mapping[e["theme"]])
        for e in p_kg
        if e["theme"] in a_p
    ]
    g_sub = [e["triplet"] for e in g_kg if e["theme"] in a_g]
    pool = _multiset(_entity_triplet_key(t) for t in g_sub)
    tp_sub = sum(_consume(pool, _entity_triplet_key(t)) for t in p_sub)

    by_pair = {}

    for t in g_tr:
        by_pair.setdefault((_norm_entity(t[0]), _norm_entity(t[3])), []).append(t)

    n_pair = rel_ok = typ_ok = both_ok = 0
    for t in p_tr:
        bucket = by_pair.get((_norm_entity(t[0]), _norm_entity(t[3])))

        if not bucket:
            continue

        g = bucket.pop()
        n_pair += 1
        r = t[2].upper() == g[2].upper()
        ty = (t[1].upper(), t[4].upper()) == (g[1].upper(), g[4].upper())
        rel_ok += r
        typ_ok += ty
        both_ok += r and ty

    return {
        "theme_exact": (tp_exact, n_p, n_g),
        "theme_structural": (len(mapping), n_p, n_g),
        "triplet_aligned": (tp_tr, len(p_tr), len(g_tr)),
        "triplet_untyped_aligned": (tp_un, len(p_tr), len(g_tr)),
        "entity_norm": (len(p_nodes & g_nodes), len(p_nodes), len(g_nodes)),
        "aligned_subgraph": (tp_sub, len(p_sub), len(g_sub)),
        "relation_given_pair": (rel_ok, n_pair),
        "types_given_pair": (typ_ok, n_pair),
        "relation_types_given_pair": (both_ok, n_pair),
        "_counts": (n_p, n_g, len(p_tr), len(g_tr)),
        "_dims": (
            [_norm(t["dimension"]) for t in (pred["themes"] if pred else [])],
            [_norm(t["dimension"]) for t in gold["themes"]],
        ),
    }


_score_example_strict = score_example
_aggregate_strict = aggregate


def score_example(pred, gold, theme_threshold=None):
    out = _score_example_strict(pred, gold, theme_threshold)
    out.update(score_staged(pred, gold))

    return out


def aggregate(per_example, n_valid_json):

    out = _aggregate_strict(per_example, n_valid_json)
    if not per_example or "theme_structural" not in per_example[0]:
        return out  # scored by the strict function alone

    for field in STAGED_PRF_FIELDS:
        tp = sum(e[field][0] for e in per_example)
        pn = sum(e[field][1] for e in per_example)
        gn = sum(e[field][2] for e in per_example)
        p, r, f = _prf(tp, pn, gn)
        out[field] = {"precision": p, "recall": r, "f1": f, "tp": tp}

    for field in CONDITIONAL_FIELDS:
        ok = sum(e[field][0] for e in per_example)
        n = sum(e[field][1] for e in per_example)
        out[field] = {"accuracy": ok / n if n else 0.0, "n": n}

    n = len(per_example)
    tp_th, tg_th, tp_tr, tg_tr = (
        sum(e["_counts"][i] for e in per_example) for i in range(4)
    )
    out["themes_per_article"] = {"pred": tp_th / n, "gold": tg_th / n}
    out["triplets_per_article"] = {"pred": tp_tr / n, "gold": tg_tr / n}

    dims = {}
    for e in per_example:
        for which, lst in zip(("pred", "gold"), e["_dims"]):
            for d in lst:
                dims.setdefault(d, {"pred": 0, "gold": 0})[which] += 1

    out["dimension_counts"] = dims

    return out


def print_staged_report(m, title="Staged evaluation"):
    print(f"=== {title} ===")
    print("STAGE 3 — joint, end to end")
    print(f"{'triplet F1 (strict)':<38} {m['triplet']['f1']:.4f}")
    print(f"{'theme F1 (char-fuzzy, strict)':<38} {m['theme']['f1']:.4f}")

    print("STAGE 1 — Theme extraction")
    for f, lab in (
        ("theme_exact", "exact string"),
        ("theme_structural", "structural alignment"),
    ):
        s = m[f]
        print(
            f"{lab:<38} P {s['precision']:.3f}  R {s['recall']:.3f}  F1 {s['f1']:.4f}"
        )

    t = m["themes_per_article"]
    print(
        f"{'themes/article pred vs gold':<38} {t['pred']:.2f} vs {t['gold']:.2f}"
        f"(recall capped at {min(1.0, t['pred'] / t['gold']):.2f})"
    )

    print("STAGE 2 — KG construction, theme wording factored out")
    for f, lab in (
        ("triplet_aligned", "triplet, aligned (5-tuple, exact)"),
        ("triplet_untyped_aligned", "triplet, aligned + untyped"),
        ("aligned_subgraph", "subgraph of aligned themes only"),
        ("entity_norm", "entity nodes (normalised)"),
    ):
        s = m[f]
        print(
            f"{lab:<38} P {s['precision']:.3f}  R {s['recall']:.3f}  F1 {s['f1']:.4f}"
        )
    print(
        f"Given the SAME (head, tail) pair as gold  [n={m['relation_given_pair']['n']:,}]"
    )

    for f, lab in (
        ("relation_given_pair", "relation correct"),
        ("types_given_pair", "both entity types correct"),
        ("relation_types_given_pair", "relation + types correct"),
    ):
        print(f"{lab:<36} {m[f]['accuracy']:.3f}")

    d = m["dimension_counts"]
    print(f"Dimension coverage")
    print(f"{'dimension':<28} {'pred':>7} {'gold':>7} {'delta':>8}")
    for k in sorted(d):
        p_, g_ = d[k]["pred"], d[k]["gold"]
        print(f"{k:<28} {p_:>7} {g_:>7} {((p_ - g_) / g_ * 100 if g_ else 0):>7.0f}%")

In [ ]:
_t_gold, _t_pred = "import tariff escalation", "trade duty escalation"


def _mk(theme):
    return [
        {
            "triplet": [theme, "CONCEPT", "HAS_ACTOR", "EU", "ORG/GOV"],
            "theme": theme,
            "dimension": "policy_or_regulation",
        },
        {
            "triplet": ["EU", "ORG/GOV", "ACTS_ON", "steel exporters", "SECTOR"],
            "theme": theme,
            "dimension": "policy_or_regulation",
        },
    ]


def _obj(kg):
    return {"knowledge_graph": kg, "themes": extract_themes(kg)}


_g, _p = _obj(_mk(_t_gold)), _obj(_mk(_t_pred))

_perfect = aggregate([score_example(_g, _g)], 1)
for _f in STAGED_PRF_FIELDS:
    assert _perfect[_f]["f1"] == 1.0, f"{_f} not 1.0 on an identical prediction"

for _f in CONDITIONAL_FIELDS:
    assert _perfect[_f]["accuracy"] == 1.0, f"{_f} not 1.0 on an identical prediction"

for _f in SCORED_FIELDS:
    assert _perfect[_f]["f1"] == 1.0, f"strict {_f} broken by the wrapper"


_renamed = aggregate([score_example(_p, _g)], 1)
assert (
    _renamed["theme_exact"]["f1"] == 0.0
), "reworded theme should not match as a string"
assert _renamed["triplet"]["f1"] < 1.0, "strict scoring should punish the rewording"
assert (
    _renamed["theme_structural"]["f1"] == 1.0
), "subgraph overlap should align the pair"
assert (
    _renamed["triplet_aligned"]["f1"] == 1.0
), "aligned scoring should recover the graph"
assert _renamed["relation_given_pair"]["accuracy"] == 1.0


for _lax, _strict in (
    ("triplet_aligned", "triplet"),
    ("triplet_untyped_aligned", "triplet_untyped"),
    ("entity_norm", "entity"),
):
    assert _renamed[_lax]["f1"] >= _renamed[_strict]["f1"] - 1e-9, f"{_lax} < {_strict}"


assert _norm_entity("Darktrace Holdings Ltd.") == _norm_entity("Darktrace")
assert _norm_entity("Nvidia Corp.") == _norm_entity("NVIDIA")
assert _norm_entity("The Fed") == _norm_entity("Fed")
assert _norm_entity("Nvidia Corp.") != _norm_entity("AMD"), "distinct companies merged!"
assert _norm_entity("Apple Inc.") != _norm_entity("Apple Records")
assert _norm_entity("Group") == "group", "normalisation must not empty a label"


print("Staged self test passed")
print(
    f"Theme reworded, graph identical; strict triplet F1"
    f" {_renamed['triplet']['f1']:.3f}, aligned {_renamed['triplet_aligned']['f1']:.3f}"
)

### Model loading + memory helpers

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def _bnb():

    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )


def load_model(adapter=None, for_training=False):

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    tok.padding_side = "right" if for_training else "left"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=_bnb(),
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    if adapter:

        from peft import PeftModel

        model = PeftModel.from_pretrained(model, adapter)

    return model, tok


def free(*objs):

    for o in objs:
        try:
            del o
        except Exception:
            pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### Evaluation loop

In [ ]:
@torch.no_grad()
def _generate(model, tok, article):

    messages = build_prompt_messages(article)
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
    )

    return tok.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


def run_eval(adapter=None, dump_predictions=None, ids=None, label=None):

    tag = label or ("fine-tuned" if adapter else "base")
    rows = load_pairs(TEST_PATH, ids=set(ids) if ids else None)
    print(f"Evaluating {tag} on {len(rows)} examples ...")

    model, tok = load_model(adapter=adapter)
    per_example, n_valid, dumps = [], 0, []
    try:
        for i, row in enumerate(rows):

            raw = _generate(model, tok, row["article"])
            pred, valid = parse_model_output(raw)
            n_valid += int(valid)
            gold = {"knowledge_graph": row["knowledge_graph"], "themes": row["themes"]}
            per_example.append(score_example(pred, gold))
            dumps.append(
                {
                    "article_id": row["article_id"],
                    "raw": raw,
                    "parsed": pred,
                    "gold": gold,
                }
            )
            print(f"  [{i+1}/{len(rows)}]", end="\r")
    finally:
        free(model, tok)

    metrics = aggregate(per_example, n_valid)
    print_report(metrics, title=tag)

    if dump_predictions:
        with open(dump_predictions, "w", encoding="utf-8") as f:
            for d in dumps:
                f.write(json.dumps(d, ensure_ascii=False) + "\n")

    return metrics


def rescore_dump(path, ids=None):

    ids = set(ids) if ids else None
    per_example, n_valid = [], 0

    with open(path, encoding="utf-8") as f:

        for line in f:
            d = json.loads(line)

            if ids is not None and d.get("article_id") not in ids:
                continue

            n_valid += int(d["parsed"] is not None)
            per_example.append(score_example(d["parsed"], d["gold"]))

    return aggregate(per_example, n_valid)

### Baseline

In [ ]:
_before_path = f"{RUN_DIR}/metrics_before.json"
if os.path.exists(_before_path):
    before_metrics = json.load(open(_before_path))
    print("Loaded cached baseline metrics - skipping re-eval.")
    print_report(before_metrics, "base model (cached)")

else:
    before_metrics = run_eval(
        adapter=None, dump_predictions=f"{RUN_DIR}/preds_before.jsonl"
    )
    save_json(before_metrics, _before_path)

### Fine tune (4 bit QLoRA)

In [ ]:
import shutil
from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from transformers import set_seed
from transformers.trainer_utils import get_last_checkpoint


def train():

    set_seed(SEED)
    train_rows = load_pairs(TRAIN_PATH)
    val_rows = load_pairs(VAL_PATH)
    train_ds = Dataset.from_list([to_chat_example(r) for r in train_rows])
    val_ds = Dataset.from_list([to_chat_example(r) for r in val_rows])

    steps_per_epoch = len(train_ds) // (BATCH_SIZE * GRAD_ACCUM)
    print(f"Train {len(train_ds):,}; Val {len(val_ds):,}")
    print(
        f"{steps_per_epoch} steps/epoch at effective batch {BATCH_SIZE * GRAD_ACCUM}"
        f"Checkpoint every {SAVE_STEPS * BATCH_SIZE * GRAD_ACCUM:,} examples"
    )

    model, tok = load_model(for_training=True)
    model.config.use_cache = False

    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=2 * LORA_R,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    cfg = SFTConfig(
        output_dir=OUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        per_device_eval_batch_size=BATCH_SIZE,
        seed=SEED,
        bf16=True,
        max_length=MAX_LENGTH,
        completion_only_loss=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=cfg,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tok,
        peft_config=peft_config,
    )

    last_ckpt = get_last_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
    if last_ckpt:
        print(f"Resuming from {last_ckpt}")

    trainer.train(resume_from_checkpoint=last_ckpt)

    trainer.save_model(OUT_DIR)
    tok.save_pretrained(OUT_DIR)

    if os.path.exists(f"{RUN_DIR}/log_history.json") and not os.path.exists(
        f"{RUN_DIR}/log_history_ep1.json"
    ):
        shutil.copy(f"{RUN_DIR}/log_history.json", f"{RUN_DIR}/log_history_ep1.json")

    save_json(trainer.state.log_history, f"{RUN_DIR}/log_history.json")
    print(f"Adapter saved to {OUT_DIR}")
    free(trainer, model, tok)


CONTINUE_TRAINING = True
if os.path.exists(f"{OUT_DIR}/adapter_model.safetensors") and not CONTINUE_TRAINING:
    print(f"Adapter already trained at {OUT_DIR}")

else:
    train()

### Score the finetuned model **after**

In [ ]:
_after_path = f"{RUN_DIR}/metrics_after.json"
if os.path.exists(_after_path):
    after_metrics = json.load(open(_after_path))
    print("Loaded cached finetuned metrics skipping re eval")
    print_report(after_metrics, "finetuned model")

else:
    after_metrics = run_eval(
        adapter=OUT_DIR, dump_predictions=f"{RUN_DIR}/preds_after.jsonl"
    )
    save_json(after_metrics, _after_path)

### Compare before v/s after

In [ ]:
def compare(before, after):

    def row(name, b, a):
        d = a - b
        arrow = "up" if d > 0 else ("down" if d < 0 else " ")
        print(f"{name:<22} {b:6.3f} {a:6.3f} {arrow:>4} {d:+.3f}")

    print(f"{'metric':<24} {'before':>6}   {'after':>6}   delta")
    print("  " + "-" * 50)

    for key in ("json_valid_rate", "ontology_valid_rate", "exact_match_rate"):
        row(key, before[key], after[key])

    for field in SCORED_FIELDS:
        row(f"{field}_f1", before[field]["f1"], after[field]["f1"])
    print()


compare(before_metrics, after_metrics)

### References:

Fine tuning code is now pretty standard and largely follows boiler plate code shown in below links. Used AI to larger degree for this notebook.

- https://huggingface.co/blog/4bit-transformers-bitsandbytes
- https://huggingface.co/docs/transformers/main_classes/quantization
- https://huggingface.co/docs/peft/en/package_reference/lora
- https://huggingface.co/docs/trl/sft_trainer
- https://huggingface.co/docs/transformers/chat_templating
- https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_clm.py